# Analisi Risultati — MAE-AST Replica
### Baade et al. (Interspeech 2022)

Questo notebook raccoglie e analizza tutti i risultati del progetto:

- **Parte A**: Pretraining su FSD50K con masking ratio 25%, 50%, 75%
  - Fine-tuning su ESC-50 per verificare il trend `acc_75 > acc_50 > acc_25`
- **Parte B**: Fine-tuning dei checkpoint pretrainati dagli autori su ESC-50
  - Target: Patch 90.0%, Frame 88.9%
- **Speedup**: Confronto tempi di training MAE-AST vs SSAST

In [ ]:
# CELLA 0 — Setup e caricamento risultati
import os, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

RESULTS_DIR = '/content/drive/MyDrive/mae_ast_results'

def load_json(name):
    path = os.path.join(RESULTS_DIR, name)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    print(f'  File non trovato: {name}')
    return None

# Carica risultati Parte A
partA_25 = load_json('partA_finetune_25pct.json')
partA_50 = load_json('partA_finetune_50pct.json')
partA_75 = load_json('partA_finetune_75pct.json')
partA_pretrain = load_json('partA_pretrain_summary.json')

# Carica risultati Parte B
partB_patch = load_json('partB_patch.json')
partB_frame = load_json('partB_frame.json')

print('Risultati caricati.')

In [ ]:
# CELLA 1 — Tabella riepilogativa completa

print('='*70)
print('  TABELLA RISULTATI COMPLETA')
print('='*70)
print(f'  {"Esperimento":<35} {"Accuracy":>12} {"Target":>10} {"Delta":>8}')
print('  ' + '-'*65)

# Parte A
print('  --- PARTE A: Pretraining FSD50K + Fine-tuning ESC-50 ---')
partA_results = []
for pct, data, label in [
    (25, partA_25, 'Masking 25%'),
    (50, partA_50, 'Masking 50%'),
    (75, partA_75, 'Masking 75%'),
]:
    if data is None:
        print(f'  {label:<35} {"N/A":>12} {"":>10} {"":>8}')
        continue
    media = data['media'] * 100
    std = data['std'] * 100
    partA_results.append((pct, media, std))
    print(f'  {label:<35} {media:.1f}+/-{std:.1f}%')

# Verifica trend
if len(partA_results) == 3:
    trend = partA_results[2][1] > partA_results[1][1] > partA_results[0][1]
    print(f'  {"":<35} Trend 75>50>25: {"SI" if trend else "NO"}')

print()
print('  --- PARTE B: Checkpoint autori + Fine-tuning ESC-50 ---')
for data, label, target in [
    (partB_patch, 'MAE-AST Patch 12L (autori)', 90.0),
    (partB_frame, 'MAE-AST Frame 12L (autori)', 88.9),
]:
    if data is None:
        print(f'  {label:<35} {"N/A":>12}')
        continue
    media = data['media'] * 100
    std = data['std'] * 100
    delta = media - target
    segno = '+' if delta >= 0 else ''
    print(f'  {label:<35} {media:.1f}+/-{std:.1f}% {target:.1f}%    {segno}{delta:.1f}%')

print('='*70)

In [ ]:
# CELLA 2 — Grafici Parte A: accuracy per fold e per masking ratio

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colori_A = {'25': '#E74C3C', '50': '#F39C12', '75': '#2E86C1'}

# --- Grafico 1: Accuracy media per masking ratio ---
ax = axes[0]
pcts = []
medie = []
errs = []
for pct, data in [(25, partA_25), (50, partA_50), (75, partA_75)]:
    if data is None:
        continue
    pcts.append(f'{pct}%')
    medie.append(data['media'] * 100)
    errs.append(data['std'] * 100)

if pcts:
    colors = [colori_A[p.replace('%','')] for p in pcts]
    bars = ax.bar(pcts, medie, yerr=errs, color=colors, alpha=0.85, capsize=5)
    for bar, m in zip(bars, medie):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{m:.1f}%', ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('Accuracy (%)')
    ax.set_xlabel('Masking Ratio')
    ax.set_title('Parte A: Accuracy vs Masking Ratio')
    ax.grid(axis='y', alpha=0.3)

# --- Grafico 2: Accuracy per fold ---
ax2 = axes[1]
x = np.arange(5)
w = 0.25
for i, (pct, data) in enumerate([(25, partA_25), (50, partA_50), (75, partA_75)]):
    if data is None:
        continue
    accs = [a * 100 for a in data['accuracies']]
    ax2.bar(x + i*w - w, accs, w, label=f'{pct}%',
            color=colori_A[str(pct)], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Parte A: Accuracy per fold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# --- Grafico 3: Pretraining loss ---
ax3 = axes[2]
if partA_pretrain:
    for k in ['mask_25', 'mask_50', 'mask_75']:
        if k in partA_pretrain:
            pct_str = k.replace('mask_', '')
            ax3.bar(f'{pct_str}%', partA_pretrain[k]['best_val_loss'],
                    color=colori_A[pct_str], alpha=0.85)
    ax3.set_ylabel('Best Validation Loss')
    ax3.set_xlabel('Masking Ratio')
    ax3.set_title('Parte A: Pretraining Loss')
    ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'analysis_partA.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELLA 3 — Grafici Parte B: confronto con paper

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Grafico 1: Accuracy per fold ---
ax = axes[0]
if partB_patch and partB_frame:
    x = np.arange(5)
    w = 0.35
    ax.bar(x - w/2, [a*100 for a in partB_patch['accuracies']],
           w, label='Patch', color='#2E75B6', alpha=0.85)
    ax.bar(x + w/2, [a*100 for a in partB_frame['accuracies']],
           w, label='Frame', color='#70AD47', alpha=0.85)
    ax.axhline(y=90.0, color='#2E75B6', linestyle='--', linewidth=1.5, label='Paper Patch')
    ax.axhline(y=88.9, color='#70AD47', linestyle='--', linewidth=1.5, label='Paper Frame')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Parte B: Accuracy per fold — ESC-50')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

# --- Grafico 2: Confronto paper ---
ax2 = axes[1]
if partB_patch and partB_frame:
    modelli = ['Paper\nPatch', 'Nostro\nPatch', 'Paper\nFrame', 'Nostro\nFrame']
    valori = [90.0, partB_patch['media']*100, 88.9, partB_frame['media']*100]
    colori = ['#2E75B6', '#4472C4', '#70AD47', '#92D050']
    bars = ax2.bar(modelli, valori, color=colori, alpha=0.85)
    for bar, val in zip(bars, valori):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom')
    ax2.set_ylim(0, 100)
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Parte B: Confronto con il paper')
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'analysis_partB.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELLA 4 — Speedup MAE-AST vs SSAST-style training
#
# MAE-AST maschera i token PRIMA dell'encoder, risparmiando FLOPs.
# SSAST (e ViT standard) passano TUTTI i token all'encoder.
#
# Speedup teorico: l'encoder processa solo (1-mask_prob) dei token.
# L'attention ha complessita O(n^2), quindi il speedup e super-lineare.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Parametri
n_patches = 256  # 8 freq x 32 time per spettrogramma [128, 512]
mask_probs = np.linspace(0, 0.9, 50)

# Speedup lineare (MLP layers)
speedup_linear = 1.0 / (1.0 - mask_probs + 1e-8)

# Speedup quadratico (attention)
n_retained = n_patches * (1.0 - mask_probs)
# FLOPs attention: O(n^2), decoder O(n_full^2) ma solo 2 layers vs 6/12
# Approssimazione: encoder domina con 6-12 layers
flops_mae = n_retained**2  # encoder
flops_full = np.full_like(mask_probs, n_patches**2)
speedup_attn = flops_full / (flops_mae + 1e-8)

ax = axes[0]
ax.plot(mask_probs * 100, speedup_linear, label='Linear layers', color='#2E86C1', linewidth=2)
ax.plot(mask_probs * 100, speedup_attn, label='Attention (O(n^2))', color='#E74C3C', linewidth=2)

# Punti per i tre masking ratio usati
for pct in [25, 50, 75]:
    mp = pct / 100
    s_lin = 1.0 / (1.0 - mp)
    s_att = n_patches**2 / (n_patches * (1-mp))**2
    ax.scatter([pct], [s_att], s=100, zorder=5, color='#E74C3C')
    ax.annotate(f'{pct}%\n{s_att:.1f}x', (pct, s_att),
                textcoords='offset points', xytext=(10, 5), fontsize=9)

ax.set_xlabel('Masking Ratio (%)')
ax.set_ylabel('Speedup vs SSAST-style')
ax.set_title('Speedup teorico MAE-AST vs SSAST')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, 90)

# Grafico accuracy vs speedup (trade-off)
ax2 = axes[1]
if partA_25 and partA_50 and partA_75:
    for pct, data, color in [
        (25, partA_25, '#E74C3C'),
        (50, partA_50, '#F39C12'),
        (75, partA_75, '#2E86C1'),
    ]:
        mp = pct / 100
        speedup = n_patches**2 / (n_patches * (1-mp))**2
        acc = data['media'] * 100
        ax2.scatter([speedup], [acc], s=150, color=color, zorder=5, label=f'{pct}%')
        ax2.annotate(f'{pct}%\n{acc:.1f}%', (speedup, acc),
                     textcoords='offset points', xytext=(10, 5), fontsize=9)

    ax2.set_xlabel('Speedup (attention)')
    ax2.set_ylabel('Accuracy ESC-50 (%)')
    ax2.set_title('Trade-off: Accuracy vs Speedup')
    ax2.legend()
    ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Dati Parte A non disponibili',
             ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'analysis_speedup.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELLA 5 — Analisi limitazioni

print('='*70)
print('  ANALISI DELLE LIMITAZIONI')
print('='*70)

limitazioni = [
    ('Dataset pretraining',
     'FSD50K (~40k clip) vs AudioSet (2M clip) usato nel paper.',
     'Le rappresentazioni apprese sono meno ricche per mancanza di dati.'),

    ('Risorse computazionali',
     'Colab A100 (1 GPU, max ~3h) vs 32 GPU per 400k steps nel paper.',
     'Il nostro pretraining usa 20k steps (1/20 del paper) con 6 encoder layers (vs 12).'),

    ('Parte B: gap ~57% con il paper',
     'I checkpoint degli autori raggiungono ~33% invece del 90% atteso.',
     'Causa: i checkpoint pubblici sono probabilmente intermedi o richiedono '
     'una configurazione di fine-tuning specifica non documentata.'),

    ('Encoder 6L vs 12L',
     'Per motivi di tempo usiamo 6 encoder layers nel pretraining.',
     'Il paper usa 12 layers per i risultati principali. '
     'Il trend masking ratio dovrebbe comunque essere visibile.'),

    ('Assenza di data augmentation nel pretraining',
     'Non usiamo mixup/cutmix durante il pretraining MAE.',
     'Il paper non specifica augmentation in pretraining, '
     'ma potrebbe essere stato usato.'),
]

for i, (titolo, causa, effetto) in enumerate(limitazioni, 1):
    print(f'\n  {i}. {titolo}')
    print(f'     Causa:   {causa}')
    print(f'     Effetto: {effetto}')

print(f'\n{"="*70}')
print('\n  CONCLUSIONE:')
print('  Nonostante le limitazioni computazionali, il progetto verifica')
print('  la tesi principale del paper: masking ratio piu alto produce')
print('  rappresentazioni migliori per la classificazione audio downstream.')
print('  Il trade-off e favorevole: 75% masking da sia il miglior speedup')
print('  (16x in attention) sia la miglior accuracy.')
print(f'{"="*70}')